# Notebook 02 — Return Construction and Statistical Diagnostics

## Objective

Notebook 01 produced a synchronized adjusted-price dataset for the seven
eligible ETFs.

This notebook converts those prices into daily returns and evaluates the
statistical properties of the return series before portfolio construction
and risk modelling.

Two return definitions are retained:

- **simple returns** for portfolio, performance and risk calculations;
- **log returns** for statistical diagnostics and dependence analysis.

The analysis examines return magnitude, volatility, skewness, kurtosis,
normality, correlation and stationarity.

Evidence of non-normality and heavy tails established here motivates the
more robust VaR, Expected Shortfall and EVT models used in later notebooks.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import jarque_bera
from statsmodels.tsa.stattools import adfuller

In [2]:
from pathlib import Path

PROJECT_ROOT = Path("..")

PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

TABLE_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "tables"
)

FIGURE_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "figures"
)

PRICE_FILE = (
    PROCESSED_DIR
    / "prices_common.csv"
)

print(
    "Project root:",
    PROJECT_ROOT.resolve()
)

print(
    "Price file:",
    PRICE_FILE.resolve()
)

print(
    "Exists:",
    PRICE_FILE.exists()
)

Project root: C:\quant_edge_round2
Price file: C:\quant_edge_round2\data\processed\prices_common.csv
Exists: True


In [3]:
prices = pd.read_csv(
    PRICE_FILE,
    index_col=0,
    parse_dates=True
)

prices = prices.sort_index()

print("Price dataset shape:", prices.shape)
print("Start date:", prices.index.min())
print("End date:", prices.index.max())

prices.head()

Price dataset shape: (5513, 7)
Start date: 2004-11-18 00:00:00
End date: 2026-10-19 00:00:00


,SPY,EFA,EEM,AGG,GLD,XLE,XLK
2004-11-18,79.547722,27.249140,13.782192,50.901527,44.380001,9.675684,7.998675
2004-11-19,78.663422,27.072721,13.585146,50.797482,44.779999,9.791416,7.874980
2004-11-22,79.038567,27.152119,13.638167,50.876785,44.950001,9.947518,7.886226
2004-11-23,79.159172,27.205036,13.661809,50.847046,44.750000,10.009421,7.852492
2004-11-24,79.346741,27.328531,13.807265,50.896587,45.049999,10.092852,7.916213


In [4]:
expected_tickers = [
    "SPY",
    "EFA",
    "EEM",
    "AGG",
    "GLD",
    "XLE",
    "XLK"
]

assert prices.shape[1] == 7

assert list(
    prices.columns
) == expected_tickers

assert prices.index.is_unique

assert (
    prices.isna()
    .sum()
    .sum()
    == 0
)

assert (
    prices.index.max()
    == pd.Timestamp(
        "2026-10-19"
    )
)

print(
    "Notebook 01 → Notebook 02 "
    "data handoff validated."
)

Notebook 01 → Notebook 02 data handoff validated.


In [5]:
simple_returns = prices.pct_change(fill_method=None).dropna()

simple_returns.head()

,SPY,EFA,EEM,AGG,GLD,XLE,XLK
2004-11-19,-0.011117,-0.006474,-0.014297,-0.002044,0.009013,0.011961,-0.015464
2004-11-22,0.004769,0.002933,0.003903,0.001561,0.003796,0.015943,0.001428
2004-11-23,0.001526,0.001949,0.001733,-0.000585,-0.004449,0.006223,-0.004278
2004-11-24,0.002370,0.004539,0.010647,0.000974,0.006704,0.008335,0.008115
2004-11-26,-0.000760,0.005811,0.016606,-0.002920,0.005327,0.004267,-0.001420


In [6]:
log_returns = np.log(
    prices / prices.shift(1)
).dropna()

log_returns.head()

,SPY,EFA,EEM,AGG,GLD,XLE,XLK
2004-11-19,-0.011179,-0.006495,-0.014400,-0.002046,0.008973,0.011890,-0.015585
2004-11-22,0.004758,0.002928,0.003895,0.001560,0.003789,0.015817,0.001427
2004-11-23,0.001525,0.001947,0.001732,-0.000585,-0.004459,0.006204,-0.004287
2004-11-24,0.002367,0.004529,0.010591,0.000974,0.006682,0.008301,0.008082
2004-11-26,-0.000760,0.005794,0.016469,-0.002924,0.005313,0.004258,-0.001421


In [7]:
print("Simple returns shape:", simple_returns.shape)
print("Log returns shape:", log_returns.shape)

print("Missing values in simple returns:")
print(simple_returns.isna().sum())

print("\nMissing values in log returns:")
print(log_returns.isna().sum())

Simple returns shape: (5512, 7)
Log returns shape: (5512, 7)
Missing values in simple returns:
SPY    0
EFA    0
EEM    0
AGG    0
GLD    0
XLE    0
XLK    0
dtype: int64

Missing values in log returns:
SPY    0
EFA    0
EEM    0
AGG    0
GLD    0
XLE    0
XLK    0
dtype: int64


In [8]:
assert simple_returns.isna().sum().sum() == 0
assert log_returns.isna().sum().sum() == 0
assert simple_returns.shape == log_returns.shape

print("Return validation passed.")

Return validation passed.


In [9]:
daily_summary = pd.DataFrame({
    "Mean Daily Return": simple_returns.mean(),
    "Daily Volatility": simple_returns.std(),
    "Minimum Daily Return": simple_returns.min(),
    "Maximum Daily Return": simple_returns.max(),
    "Skewness": simple_returns.skew(),
    "Excess Kurtosis": simple_returns.kurt()
})

daily_summary

,Mean Daily Return,Daily Volatility,Minimum Daily Return,Maximum Daily Return,Skewness,Excess Kurtosis
SPY,0.000482,0.011858,-0.109424,0.145198,-0.001490,15.523296
EFA,0.000328,0.013166,-0.111632,0.158876,-0.039746,14.298407
EEM,0.000427,0.017358,-0.161662,0.227699,0.496887,17.803153
AGG,0.000119,0.003273,-0.068395,0.038687,-1.838344,50.605483
GLD,0.000456,0.011506,-0.102742,0.112905,-0.306334,6.407324
XLE,0.000529,0.018673,-0.201412,0.164746,-0.316692,11.625517
XLK,0.000672,0.014219,-0.138140,0.138983,0.087281,9.785572


In [10]:
TRADING_DAYS = 252

annualized_summary = pd.DataFrame({
    "Annualized Mean Return": simple_returns.mean() * TRADING_DAYS,
    "Annualized Volatility": simple_returns.std() * np.sqrt(TRADING_DAYS)
})

annualized_summary

,Annualized Mean Return,Annualized Volatility
SPY,0.121352,0.188233
EFA,0.082674,0.209001
EEM,0.107582,0.275545
AGG,0.029930,0.051950
GLD,0.114897,0.182655
XLE,0.133321,0.296432
XLK,0.169361,0.225722


In [11]:
years = (
    prices.index[-1] - prices.index[0]
).days / 365.25

cagr = (
    prices.iloc[-1] / prices.iloc[0]
) ** (1 / years) - 1

annualized_summary["CAGR"] = cagr

annualized_summary

,Annualized Mean Return,Annualized Volatility,CAGR
SPY,0.121352,0.188233,0.108934
EFA,0.082674,0.209001,0.062557
EEM,0.107582,0.275545,0.072083
AGG,0.029930,0.051950,0.028927
GLD,0.114897,0.182655,0.102923
XLE,0.133321,0.296432,0.093014
XLK,0.169361,0.225722,0.154389


In [12]:
jb_rows = []

for ticker in log_returns.columns:
    jb_stat, jb_p = jarque_bera(log_returns[ticker])

    jb_rows.append({
        "Ticker": ticker,
        "JB Statistic": jb_stat,
        "p-value": jb_p,
        "Normal at 5%": jb_p >= 0.05
    })

jb_summary = pd.DataFrame(jb_rows)

jb_summary

,Ticker,JB Statistic,p-value,Normal at 5%
0,SPY,50662.553907,0.0,False
1,EFA,43118.403027,0.0,False
2,EEM,55592.004692,0.0,False
3,AGG,708417.562985,0.0,False
4,GLD,10075.569465,0.0,False
5,XLE,38796.201595,0.0,False
6,XLK,20557.886070,0.0,False


## Normality Interpretation

The Jarque–Bera test rejects normality at the 5% significance level for all
seven ETF return series.

The return distributions also exhibit substantial excess kurtosis, indicating
fatter tails than the normal distribution.

This result is important for later risk modelling because parametric
normal-distribution VaR may underestimate extreme losses.

Historical VaR, Expected Shortfall and EVT are therefore retained as
alternative downside-risk measures in later notebooks.

In [13]:
for ticker in simple_returns.columns:
    plt.figure(figsize=(8, 4))
    plt.hist(
        simple_returns[ticker],
        bins=60
    )
    plt.title(f"{ticker} Daily Return Distribution")
    plt.xlabel("Daily Return")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

C:\Users\Jani\AppData\Local\Temp\ipykernel_24640\1926765897.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
correlation_matrix = log_returns.corr()

correlation_matrix

,SPY,EFA,EEM,AGG,GLD,XLE,XLK
SPY,1.000000,0.880719,0.817254,0.002219,0.067743,0.704763,0.906845
EFA,0.880719,1.000000,0.868533,0.039718,0.180288,0.686877,0.767421
EEM,0.817254,0.868533,1.000000,-0.008503,0.195991,0.654174,0.743001
AGG,0.002219,0.039718,-0.008503,1.000000,0.208542,-0.051570,-0.002264
GLD,0.067743,0.180288,0.195991,0.208542,1.000000,0.147164,0.063545
XLE,0.704763,0.686877,0.654174,-0.051570,0.147164,1.000000,0.537624
XLK,0.906845,0.767421,0.743001,-0.002264,0.063545,0.537624,1.000000


In [15]:
plt.figure(figsize=(8, 6))

plt.imshow(
    correlation_matrix,
    aspect="auto"
)

plt.xticks(
    range(len(correlation_matrix.columns)),
    correlation_matrix.columns,
    rotation=45
)

plt.yticks(
    range(len(correlation_matrix.index)),
    correlation_matrix.index
)

plt.colorbar(label="Correlation")

plt.title("ETF Return Correlation Matrix")
plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "correlation_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

C:\Users\Jani\AppData\Local\Temp\ipykernel_24640\3558608762.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Diversification Interpretation

The equity-oriented ETFs display relatively high positive correlations,
particularly SPY with XLK and SPY with EFA.

By contrast, AGG has very low correlation with the equity assets, while GLD
also exhibits substantially lower correlations than the major equity ETFs.

These relationships suggest that bonds and gold may provide diversification
benefits in portfolio construction.

The correlation analysis therefore provides the first empirical justification
for considering multi-asset portfolios rather than concentrating solely in
the highest-return assets.

In [16]:
rolling_volatility = (
    log_returns
    .rolling(window=20)
    .std()
    * np.sqrt(252)
)

In [17]:
plt.figure(figsize=(12, 6))

for ticker in rolling_volatility.columns:
    plt.plot(
        rolling_volatility.index,
        rolling_volatility[ticker],
        label=ticker
    )

plt.title("20-Day Rolling Annualized Volatility")
plt.xlabel("Date")
plt.ylabel("Annualized Volatility")
plt.legend()
plt.tight_layout()

plt.savefig(
    FIGURE_DIR / "rolling_volatility_20d.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

C:\Users\Jani\AppData\Local\Temp\ipykernel_24640\2513005942.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [18]:
adf_rows = []

for ticker in log_returns.columns:
    result = adfuller(
        log_returns[ticker].dropna(),
        autolag="AIC"
    )

    adf_rows.append({
        "Ticker": ticker,
        "ADF Statistic": result[0],
        "p-value": result[1],
        "Lags Used": result[2],
        "Observations": result[3],
        "Stationary at 5%": result[1] < 0.05
    })

adf_summary = pd.DataFrame(adf_rows)

adf_summary

,Ticker,ADF Statistic,p-value,Lags Used,Observations,Stationary at 5%
0,SPY,-16.706634,1.446186e-29,19,5492,True
1,EFA,-16.218449,3.920344e-29,19,5492,True
2,EEM,-16.255691,3.608003e-29,19,5492,True
3,AGG,-15.815845,1.033780e-28,21,5490,True
4,GLD,-75.421742,0.000000e+00,0,5511,True
5,XLE,-18.185065,2.427416e-30,17,5494,True
6,XLK,-18.463863,2.144772e-30,15,5496,True


## Stationarity Interpretation

The Augmented Dickey-Fuller test rejects the unit-root null hypothesis for all
seven log-return series at the 5% significance level.

The return series are therefore treated as stationary over the study period
for the subsequent statistical and portfolio analysis.

This result applies to returns, not to the underlying price levels.

In [19]:
assert (
    simple_returns.shape[0]
    == prices.shape[0] - 1
)

assert (
    log_returns.shape
    == simple_returns.shape
)

assert (
    simple_returns.index.equals(
        log_returns.index
    )
)

assert (
    simple_returns.isna()
    .sum()
    .sum()
    == 0
)

assert (
    log_returns.isna()
    .sum()
    .sum()
    == 0
)

assert (
    adf_summary[
        "Stationary at 5%"
    ].all()
)

assert (
    ~jb_summary[
        "Normal at 5%"
    ].any()
)

print(
    "All Notebook 02 final "
    "validation checks passed."
)

All Notebook 02 final validation checks passed.


In [20]:
simple_returns.to_csv(
    PROCESSED_DIR
    / "simple_returns.csv"
)

log_returns.to_csv(
    PROCESSED_DIR
    / "log_returns.csv"
)

daily_summary.to_csv(
    TABLE_DIR
    / "daily_return_summary.csv"
)

annualized_summary.to_csv(
    TABLE_DIR
    / "annualized_return_summary.csv"
)

jb_summary.to_csv(
    TABLE_DIR
    / "jarque_bera_normality_summary.csv",
    index=False
)

correlation_matrix.to_csv(
    TABLE_DIR
    / "correlation_matrix.csv"
)

adf_summary.to_csv(
    TABLE_DIR
    / "adf_stationarity_summary.csv",
    index=False
)

print(
    "Notebook 02 outputs saved."
)

Notebook 02 outputs saved.
